# 02. 2:4 Semi-Structured Sparsity and RigL Dynamic Sparse Training

- **Category:** 15_compression_peft_and_speculative_inference / 01_pruning_sparsity_lowrank_and_distillation
- **Lineage & Citations:** Mishra et al. (2021, *Accelerating Sparse Deep Neural Networks*, arXiv:2104.08378); Evci et al. (2020, *Rigging the Lottery: Making Trainable Sparse Networks*, ICML).
- **Purpose:** Formulate hardware-aligned 2:4 semi-structured sparsity (2 zeroes in every 4 contiguous elements) for 2x Sparse Tensor Core throughput, alongside RigL dynamic sparse training via weight eviction and gradient-guided regrowth under a fixed parameter budget.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Unstructured, Structured Pruning, ShortGPT, and the Lottery Ticket Hypothesis](01_unstructured_structured_shortgpt_and_lottery_ticket.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. LLM One-Shot Pruning: Wanda and SparseGPT](03_llm_oneshot_pruning_wanda_and_sparsegpt.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT deploy 2:4 sparsity without dedicated Sparse Tensor Core hardware support (on GPUs lacking 2:4 Sparse MMA units, overhead cancels gains); do NOT train RigL without decaying the drop fraction $\alpha_t$ toward zero (perpetual high churn prevents convergence near optima).

## Mathematical Foundations & Governing Equations

1. **2:4 Block Selection Rule**:
   $$M_{r, 4b + j} = \mathbb{I}\left( j \in \arg\text{top2}_{k \in \{0,1,2,3\}} |W_{r, 4b + k}| \right), \quad W_{2:4} = W \odot M$$
2. **2-Bit Metadata Storage Compression**:
   $$\text{Bits per 4-block} = 2 \times 16\text{ bits (data)} + 4\text{ bits (index)} = 36\text{ bits vs } 64\text{ bits (dense)}$$
3. **RigL Drop & Regrow Rule**:
   $$\mathcal{D}_t = \arg\min_{|\mathcal{D}|=k} \sum_{(i,j) \in \mathcal{D} \subset \mathcal{S}_t} |W_{ij}|, \quad \mathcal{R}_t = \arg\max_{|\mathcal{R}|=k} \sum_{(i,j) \in \mathcal{R} \subset \mathcal{S}_t^c} |\nabla_{W_{ij}} \mathcal{L}|$$
4. **RigL Cosine Drop Schedule**:
   $$\alpha_t = \frac{\alpha_0}{2} \left( 1 + \cos\left(\frac{\pi t}{T_{\text{end}}}\right) \right)$$

### Asymptotic Complexity
- 2:4 Pruning: $\mathcal{O}(M \cdot K)$ linear block sweep, zero auxiliary dynamic allocations.
- RigL Topology Update: $\mathcal{O}(N \log k)$ top-$k$ selection over active and inactive sets.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} golden tensors from {WEIGHTS_PATH.name}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def prune_2_to_4(w: torch.Tensor):
    rows, cols = w.shape
    w_blocks = w.view(rows, cols // 4, 4)
    abs_blocks = torch.abs(w_blocks)
    top2_indices = torch.topk(abs_blocks, k=2, dim=-1, largest=True).indices
    mask_blocks = torch.zeros_like(w_blocks)
    mask_blocks.scatter_(-1, top2_indices, 1.0)
    mask = mask_blocks.view(rows, cols)
    return w * mask, mask

def compress_2_to_4(w: torch.Tensor, mask: torch.Tensor):
    rows, cols = w.shape
    num_blocks = cols // 4
    w_blocks = w.view(rows, num_blocks, 4)
    mask_blocks = mask.view(rows, num_blocks, 4)
    vals_list, idxs_list = [], []
    for r in range(rows):
        row_vals, row_idxs = [], []
        for b in range(num_blocks):
            kept = torch.nonzero(mask_blocks[r, b] > 0.5, as_tuple=False).flatten()
            for idx in kept:
                row_vals.append(w_blocks[r, b][idx])
                row_idxs.append(idx.to(torch.float32))
        vals_list.append(torch.stack(row_vals))
        idxs_list.append(torch.stack(row_idxs))
    return torch.stack(vals_list), torch.stack(idxs_list)

def rigl_update_mask(w: torch.Tensor, grad: torch.Tensor, current_mask: torch.Tensor, drop_fraction: float = 0.25):
    flat_w, flat_grad, flat_mask = w.flatten(), grad.flatten(), current_mask.flatten()
    active = torch.nonzero(flat_mask > 0.5, as_tuple=False).flatten()
    inactive = torch.nonzero(flat_mask <= 0.5, as_tuple=False).flatten()
    k_drop = max(1, int(drop_fraction * len(active)))
    drop_sub = torch.topk(torch.abs(flat_w[active]), k_drop, largest=False).indices
    regrow_sub = torch.topk(torch.abs(flat_grad[inactive]), k_drop, largest=True).indices
    new_mask = flat_mask.clone()
    new_mask[active[drop_sub]] = 0.0
    new_mask[inactive[regrow_sub]] = 1.0
    return new_mask.view_as(current_mask)

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-6

# 2. 2:4 Semi-structured pruning parity
w_24, mask_24 = prune_2_to_4(weights["w"])
assert float(torch.max(torch.abs(w_24 - weights["w_2_to_4"]))) < 1e-6
assert float(torch.max(torch.abs(mask_24 - weights["mask_2_to_4"]))) < 1e-6

# 3. 2:4 Compressed representation parity
vals_24, idxs_24 = compress_2_to_4(weights["w"], mask_24)
assert float(torch.max(torch.abs(vals_24 - weights["packed_values"]))) < 1e-6
assert float(torch.max(torch.abs(idxs_24 - weights["packed_indices"]))) < 1e-6

# 4. RigL dynamic topology update parity
mask_s1 = rigl_update_mask(weights["w"], weights["grad"], weights["initial_mask"], drop_fraction=0.25)
assert float(torch.max(torch.abs(mask_s1 - weights["rigl_mask_step1"]))) < 1e-6
print("All 4 2:4 and RigL implementations verified against seed-42 golden tensors!")

In [ ]:
# Empirical evaluation: 2:4 Structure Verification & Storage Savings
mask_blocks = mask_24.view(16, 4, 4)
block_sums = mask_blocks.sum(dim=-1)
assert torch.all(block_sums == 2.0), "Every 4-element block must contain exactly two non-zeroes!"
dense_elements = weights["w"].numel()
packed_elements = vals_24.numel()
print(f"Original dense matrix elements: {dense_elements}")
print(f"Packed 2:4 value elements:     {packed_elements} ({packed_elements/dense_elements*100:.1f}% storage)")
print(f"Metadata indices range:        [{idxs_24.min():.0f}, {idxs_24.max():.0f}] (fits in 2 bits per value!)")
print(f"RigL active connections count:  {int(mask_s1.sum())}/{mask_s1.numel()} (preserved constant!)")

In [ ]:
# Pathological Misuse: Random Sparsity vs Structured 2:4 Sparsity
# Random 50% sparsity breaks contiguous 4-element Sparse Tensor Core alignments;
# 2:4 sparsity guarantees exactly 2 values per 4-vector across all rows.
random_mask = (torch.rand(16, 16) > 0.5).to(torch.float32)
rand_block_sums = random_mask.view(16, 4, 4).sum(dim=-1)
violations = int((rand_block_sums != 2.0).sum())
print(f"2:4 Hardware alignment violations in random 50% mask: {violations}/{rand_block_sums.numel()} blocks")
assert violations > 0, "Random mask should almost certainly violate 2:4 hardware constraints!"
print("Pathology demonstrated: Random sparsity cannot execute on Sparse Tensor Cores without 2:4 pattern enforcement.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Hardware-Enforced Sparsity**: 2:4 semi-structured sparsity guarantees exactly 50% sparsity with hardware index multiplexing, unlocking 2x arithmetic GEMM speedup on NVIDIA Ampere/Hopper/Blackwell Tensor Cores.
2. **Dynamic RigL Training**: Dynamic sparse training with gradient regrowth allows exploring sparse network topologies from scratch without requiring dense pretraining or multi-cycle iterative magnitude pruning.
3. **Metadata Storage**: Retaining 2 values per 4 elements requires only 4 bits of metadata per block, compressing total storage to ~56% of full dense precision.

---
### Cross-Domain Generalisation Matrix
- **Quantization**: 2:4 metadata packing is isomorphic to sub-byte lookup table (LUT) index packing in NF4 and additive quantization.
- **Distributed**: RigL dynamic topology reallocation mirrors communication-avoiding dynamic all-reduce topologies in distributed SGD.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.md](../../../markdown/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.md)
- **PyTorch 2.x Module:** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.py](../../../pytorch2/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.py)
- **Functional JAX Module:** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.py](../../../jax/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.py)
- **Native Rust Candle Track:** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.rs](../../../candle/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.rs)
- **Hardware Kernel (CPU / WGSL):** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.rs](../../../kernels/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.rs)
- **Golden SafeTensors:** [02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.safetensors](../../../weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/02_2_4_semi_structured_sparsity_and_rigl_dynamic_sparse.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Unstructured Structured Shortgpt and Lottery Ticket](01_unstructured_structured_shortgpt_and_lottery_ticket.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 03. LLM Oneshot Pruning Wanda and Sparsegpt](03_llm_oneshot_pruning_wanda_and_sparsegpt.ipynb) &rarr;